# 1. Exploratory Data Analysis

Overview of group composition, conversion rates, and ad exposure patterns across the Advertisement (ad) and Public Service Announcement (psa) groups prior to hypothesis testing.

## Load Data

In [ ]:
# import kaggle
# kaggle.api.dataset_download_files(
#     'faviovaz/marketing-ab-testing',
#     path='.',
#     unzip=True
# )

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

path = "ab_marketing/marketing_ab.csv"
df = pd.read_csv(path)

In [ ]:
df.head()

In [ ]:
try:
    df.drop(columns=["Unnamed: 0"], inplace=True)
    df.rename(columns={
        'user id': 'user_id',
        'test group': 'test_group',
        'total ads': 'total_ads',
        'most ads day': 'most_ads_day',
        'most ads hour': 'most_ads_hour'
    }, inplace=True)
except KeyError as k:
    print('Cell already run.')

In [ ]:
df['test_group'].unique()

In [ ]:
print(df.shape[0])
df.head()

## Data Validation

In [ ]:
print(f'Data Types: \n\
      {df.dtypes}')

In [ ]:
print(f'Missing Values: \n\
      \n\
      {df.isnull().sum()}')

In [ ]:
print(f'Duplicated values: {df.duplicated().sum()}')

There are not duplicate of missing values in the dataset.

## Descriptive Statistics

In [ ]:
print('Numeric Summary')
display(
    df[['total_ads', 'most_ads_hour']]
    .describe()
    .round(2)
)

In [ ]:
print('Group Sizes')
print(df['test_group'].value_counts())
print(df['test_group'].value_counts(normalize=True))

There group exposed to **ads** is significantly larger (96%) than the one exposed to **psa** (4%).

In [ ]:
print('Conversion Rate Overall')
print(df['converted'].value_counts(normalize=True).round(4))

The overall **conversion rate** is ~2.5%

In [ ]:
print('Conversion Rate By Group')
print(df.groupby('test_group')['converted'].mean().round(4))

The conversion rate was ~2.5% for **ads** and ~1.8% for **psa**.

## Class Imbalance Chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['converted'].value_counts()
axes[0].bar(['Not Converted', 'Converted'], counts.values, color=['#4C72B0', '#DD8452'])
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)
    axes[0].set_title('Overall Conversion')
    axes[0].set_ylabel('Count')
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

group_conv = df.groupby('test_group')['converted'].mean().reset_index()
axes[1].bar(group_conv['test_group'], group_conv['converted'] * 100, color=['#4C72B0', '#DD8452'])
for i, row in group_conv.iterrows():
    axes[1].text(i, row['converted'] * 100 + 0.05, f"{row['converted']*100:.2f}%", ha='center', fontsize=10)
    axes[1].set_title('Conversion Rate by Group')
    axes[1].set_ylabel('Conversion Rate (%)')
    axes[1].set_ylim(0, group_conv['converted'].max() * 100 * 1.3)

plt.suptitle('Conversion Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Overall conversion is very low - only 2.5% of the 588,101 users converted, leaving 97.5% unconverted. This is typical for digital marketing campaigns and sets a low baseline to beat.

When split by group, the ad group converts at 2.55% vs. 1.79% for the PSA group - a 0.76% absolute difference and roughly a 42% relative lift. The direction is in favor of the ads, but whether this gap is meaningful needs statistical confirmation.

## Total Ads Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df['total_ads'], bins=100, color='#4C72B0', edgecolor='white', linewidth=0.3)
axes[0].set_title('total_ads — Full Range')
axes[0].set_xlabel('Total Ads')
axes[0].set_ylabel('Count')

p99 = df['total_ads'].quantile(0.99)
axes[1].hist(df.loc[df['total_ads'] <= p99, 'total_ads'], bins=60, color='#4C72B0', edgecolor='white', linewidth=0.3)
axes[1].set_title(f'total_ads — Clipped at p99 ({p99:.0f})')
axes[1].set_xlabel('Total Ads')

df_clip = df[df['total_ads'] <= p99]
groups = [df_clip.loc[df_clip['test_group'] == g, 'total_ads'].values for g in ['ad', 'psa']]
axes[2].boxplot(groups, labels=['ad', 'psa'], patch_artist=True,
                boxprops=dict(facecolor='#4C72B080'),
                medianprops=dict(color='#DD8452', linewidth=2))
axes[2].set_title('total_ads by Group (clipped p99)')
axes[2].set_ylabel('Total Ads')

print(f'Skewness: {df["total_ads"].skew():.2f}')
print(df['total_ads'].quantile([.25,.5,.75,.9,.95,.99]).round(1))

plt.suptitle('total_ads Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

The distribution is heavily right-skewed - the vast majority of users saw very few ads, with the bulk concentrated below 25, while a long tail extends all the way to 2,065. Clipping at p99 (202 ads) makes the shape clearer: most exposure is low, but there's a meaningful segment of heavy viewers.

The boxplot by group shows nearly identical distributions between ads and psa users — similar medians, similar spread, and comparable upper tails.

## Conversion by Day

In [ ]:
import numpy as np

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

day_conv = (
    df.groupby(['most_ads_day', 'test_group'])['converted']
    .mean().reset_index()
)
day_conv['most_ads_day'] = pd.Categorical(day_conv['most_ads_day'], categories=day_order, ordered=True)
day_conv = day_conv.sort_values('most_ads_day')

fig, ax = plt.subplots(figsize=(11, 5))
width = 0.35
x = np.arange(len(day_order))

for i, (group, color) in enumerate(zip(['ad', 'psa'], ['#4C72B0', '#DD8452'])):
    vals = day_conv[day_conv['test_group'] == group].set_index('most_ads_day').reindex(day_order)['converted'].values * 100
    ax.bar(x + i * width - width/2, vals, width, label=group, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(day_order)
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Conversion Rate by Day of Week', fontsize=13, fontweight='bold')
ax.legend(title='Group')
plt.tight_layout()
plt.show()

The ads group consistently outperforms the PSA group across every day of the week. Monday and Tuesday stand out as the strongest days for the ads group (~3.3% and ~3.0%), while Thursday and Saturday are the weakest (~2.2%).

The PSA group follows a broadly similar pattern but at a lower level throughout, with Tuesday and Saturday being its weakest days (~1.5% and ~1.4%).

## Conversion by Hour

In [ ]:
hour_conv = (
    df.groupby(['most_ads_hour', 'test_group'])['converted']
    .mean().reset_index()
)

fig, ax = plt.subplots(figsize=(13, 5))
for group, color in zip(['ad', 'psa'], ['#4C72B0', '#DD8452']):
    subset = hour_conv[hour_conv['test_group'] == group].sort_values('most_ads_hour')
    ax.plot(subset['most_ads_hour'], subset['converted'] * 100,
            marker='o', label=group, color=color, linewidth=2, markersize=5)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Conversion Rate by Hour of Day', fontsize=13, fontweight='bold')
ax.set_xticks(range(0, 24))
ax.legend(title='Group')
plt.tight_layout()
plt.show()

Both groups show a clear upward trend throughout the day, with conversion rates rising steadily from morning into the afternoon and evening, peaking around hours 15–21 before declining late at night.

The **ads** group climbs from roughly 1.5% in the early hours to a peak of ~3.1% around hour 16. The **psa** group shows the same general trend but with high volatility in the early morning hours (0–4 AM), likely due to its much smaller sample size. From the afternoon onward, it tracks the ads group's shape but at a consistently lower level.

## Dose Response

In [ ]:
ad_group = df[df['test_group'] == 'ad'].copy()
p99 = ad_group['total_ads'].quantile(0.99)
ad_clip = ad_group[ad_group['total_ads'] <= p99].copy()

ad_clip['ads_bin'] = pd.qcut(ad_clip['total_ads'], q=10, duplicates='drop')
dose = ad_clip.groupby('ads_bin', observed=True)['converted'].agg(['mean', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(range(len(dose)), dose['mean'] * 100, color='#4C72B0', alpha=0.8)
ax.set_xticks(range(len(dose)))
ax.set_xticklabels([str(b) for b in dose['ads_bin']], rotation=35, ha='right', fontsize=8)
ax.set_xlabel('total_ads (decile bins)')
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Dose-Response: Ads Seen vs Conversion Rate (ads group)', fontsize=13, fontweight='bold')
for i, row in dose.iterrows():
    ax.text(i, row['mean'] * 100 + 0.02, f"n={row['count']:,}", ha='center', fontsize=7, color='gray')
plt.tight_layout()
plt.show()

There is a clear positive dose-response relationship in the ads group — conversion rate rises steadily with the number of ads seen. Users in the lowest exposure bin (1–2 ads) convert at near 0%, while those in the highest bin (53–201 ads) reach ~14%, more than five times the group's overall average.

The relationship is non-linear: conversion stays very low through the first several bins and only begins to accelerate meaningfully beyond 23 ads, suggesting a threshold effect where repeated exposure is needed before ads start driving conversions.

## Randomization Check

In [ ]:
from scipy.stats import mannwhitneyu, chi2_contingency

ad = df[df['test_group'] == 'ad']
psa = df[df['test_group'] == 'psa']

stat, p = mannwhitneyu(ad['total_ads'], psa['total_ads'], alternative='two-sided')
print(f'total_ads — Mann-Whitney U: p={p:.4f}')

chi2, p_day, dof, _ = chi2_contingency(pd.crosstab(df['test_group'], df['most_ads_day']))
print(f'most_ads_day — Chi²={chi2:.2f}, p={p_day:.4f}')

chi2_h, p_hour, dof_h, _ = chi2_contingency(pd.crosstab(df['test_group'], df['most_ads_hour']))
print(f'most_ads_hour — Chi²={chi2_h:.2f}, p={p_hour:.4f}')

All three p-values are ~0, meaning the groups are statistically different on every pre-treatment variable. The null hypothesis for each test is **"the two groups have the same distribution of this variable."**

* Users in **ads** group and **psa** group saw different volumes of ads.
* They were active on different days.
* They were active at different hours.

These variables aren't outcomes, but characteristics of users before (or independent of) conversion. If the groups differ on them, the assignment wasn't perfectly random, or there's a structural reason users ended up in one group vs the other.

The concern is that if **ads** users happen to be more active (more total ads, peak-hours with higher intent), some of the conversion lift observed might be due to that **pre-existing difference**, not due to the ad itself.

In [ ]:
print(df.groupby('test_group')['total_ads'].median())
print(df.groupby('test_group')['most_ads_hour'].value_counts(normalize=True).unstack())

Randomization is functional, the groups are balanced in a meaningful sense. The failed statistical tests are a well-known large-N phenomenon - with enough observations, even a difference of 1 median ad or 0.005 proportion diference will produce P=0.

## Summary Table

In [ ]:
summary = df.groupby('test_group').agg(
    n=('user_id', 'count'),
    conversions=('converted', 'sum'),
    conversion_rate=('converted', 'mean'),
    avg_total_ads=('total_ads', 'mean'),
    median_total_ads=('total_ads', 'median'),
).round(4)
summary['conversion_rate_%'] = (summary['conversion_rate'] * 100).round(2)
display(summary)

The **ads** group is 24x larger than the **psa** group and converts at 2.55% vs. 1.79% — a 0.76 pp absolute difference that will be tested for statistical  significance in the next notebook.